# Module 6: Automation & Process Management
In this notebook, we will simulate how professional automation pipelines work. 

First, we will build a "Dummy C++ Vision Tool" (written in Python for this example) that simulates analyzing an image. Then, we will use the `subprocess` module to control this tool from the outside.

In [2]:
%%writefile vision_binary.py
import sys
import time
import argparse

# Simulating an external, compiled program that we cannot modify
parser = argparse.ArgumentParser()
parser.add_argument("--image", required=True, help="Image to process")
parser.add_argument("--fail", action="store_true", help="Simulate a crash")
args = parser.parse_args()

print(f"[VISION TOOL] Loading {args.image} into memory...")
time.sleep(1) # Simulating processing time

if args.fail:
    # Print the error to standard error (stderr)
    print(f"[VISION TOOL] FATAL ERROR: Corrupted image data in {args.image}", file=sys.stderr)
    # Exit with a non-zero code to tell the OS we failed!
    sys.exit(1)
else:
    # Print success to standard output (stdout)
    print(f"[VISION TOOL] SUCCESS: 3 Objects detected in {args.image}")
    # Exit with a 0 to tell the OS everything is fine
    sys.exit(0)

Writing vision_binary.py


## Capturing Output and Checking Exit Codes
Now we write our Master Automation Script. It will use `subprocess` to run `vision_binary.py` via the terminal. It will capture the output and check the Return Code to decide what to do next.

*Scenario 1: We send a valid image.*

In [3]:
import subprocess

print("=== 1. Automating a Successful Process ===")

# We pass the command as a list of strings, exactly how we would type it in the terminal
command = ["python", "vision_binary.py", "--image", "thermal_scan_01.jpg"]

print(f"Executing external command: {' '.join(command)}\n")

# Run the command and capture the output!
result = subprocess.run(command, capture_output=True, text=True)

# 1. Check the Exit Code
print(f"Exit Code: {result.returncode}")

if result.returncode == 0:
    print("✅ System confirms the external process completed successfully.")
    print("Captured Output from the tool:")
    print("-" * 30)
    print(result.stdout.strip())
    print("-" * 30)
else:
    print("❌ Process Failed!")

=== 1. Automating a Successful Process ===
Executing external command: python vision_binary.py --image thermal_scan_01.jpg

Exit Code: 0
✅ System confirms the external process completed successfully.
Captured Output from the tool:
------------------------------
[VISION TOOL] Loading thermal_scan_01.jpg into memory...
[VISION TOOL] SUCCESS: 3 Objects detected in thermal_scan_01.jpg
------------------------------


*Scenario 2: We send a corrupted image.*

Our Master Script needs to handle crashes gracefully. If the external tool fails, our entire pipeline shouldn't explode. We check the exit code, log the error from `stderr`, and move on safely.

In [4]:
print("=== 2. Handling External Process Failures ===")

# This time, we add the --fail flag to simulate a corrupted file crash
bad_command = ["python", "vision_binary.py", "--image", "corrupted_scan.jpg", "--fail"]

print(f"Executing external command: {' '.join(bad_command)}\n")

# Run the command and capture the output
bad_result = subprocess.run(bad_command, capture_output=True, text=True)

# 1. Check the Exit Code
print(f"Exit Code: {bad_result.returncode}")

if bad_result.returncode != 0:
    print("⚠️ WARNING: The external tool crashed!")
    # When programs crash, they usually print their errors to stderr, not stdout
    print("\nExtracting Crash Logs (stderr):")
    print("-" * 30)
    print(bad_result.stderr.strip())
    print("-" * 30)
    print("\nOur Master Script caught the error and remains online. Pipeline safely stopped.")

=== 2. Handling External Process Failures ===
Executing external command: python vision_binary.py --image corrupted_scan.jpg --fail

Exit Code: 1
⚠️ WARNING: The external tool crashed!

Extracting Crash Logs (stderr):
------------------------------
[VISION TOOL] FATAL ERROR: Corrupted image data in corrupted_scan.jpg
------------------------------

Our Master Script caught the error and remains online. Pipeline safely stopped.
